In [ ]:
# Пиксели исходного изображения рассматриваются как настраиваемые праметры,
# которые подбираются с помощью градиентного спуска
#
# В стилизации Нейросеть используется для вычисления критерия качества
#
# Общий критерий качества состоит из степени соответствия и степени стилизации
#
# Для стилизации будем использовать сеть VGG-19
#
# Процесс стилизации сопровождается вичислением матрицы Грамма на каждом
# сверточном слое VGG-19

In [ ]:
import torch
import torch.nn as nn
import torch.utils.data as data
import torchvision.transforms.v2 as tfs_v2
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from torchvision import models
from tqdm import tqdm
from PIL import Image

class ModelStyle(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        _model = models.vgg19(weights=models.VGG19_Weights.DEFAULT) # Обычная vgg-19
        self.mf = _model.features                                   # Выделяет только сверточные обучаемые слои
        self.mf.requires_grad_(False)                               # Нейросеть не обучается
        self.requires_grad_(False)                                  # Нейросеть не обучается
        self.mf.eval()                                              # Режим эксплуатации
        self.idx_out = (0, 5, 10, 19, 28, 34)                       # Индексы свёрточных слоев из документации
        self.num_style_layers = len(self.idx_out) - 1               # Последний слой

    def forward(self, x):
        outputs = []
        for idx, layer in enumerate(self.mf):
            x = layer(x) # Но здесь ещё не матрица Грамма
            if idx in self.idx_out:
                outputs.append(x.squeeze(0))
        return outputs

# Матрица Грамма
def gram_matrix(x):          # Из тензора (512, 14, 14) формируем матрицу (512, 14x14)
    channels = x.size(dim=0) # Число каналов
    g = x.view(channels, -1) # Каждый из признаков вытягивается в вектор
    # Матричное умножение матрицы на ее транспонированную версию и делиться на число признаков
    gram = torch.mm(g, g.mT) / g.size(dim=1)
    return gram

# Потеря по контенту
def get_content_loss(base_content, target):
    return torch.mean(torch.square(base_content - target))

# Потеря по стилю
def get_style_loss(base_style, gram_target):
    style_weights = [1.0, 0.8, 0.5, 0.3, 0.1]
    _loss = 0
    i = 0
    for base, target in zip(base_style, gram_target):
        gram_style = gram_matrix(base)
        _loss += style_weights[i] * torch.mean(torch.square(gram_style-target))
        i += 1
    return _loss

img = Image.open('img.jpg').convert('RGB')
img_style = Image.open('img_style.jpg').convert('RGB')

transforms = tfs_v2.Compose([tfs_v2.ToImage,
                             tfs_v2.ToDtype(torch.float32, scale=True)])

img = transforms(img).unsqueeze(0)
img_style = transforms(img_style).unsqueeze(0)
img_create = img.clone()
img_create.requires_grad_(True)

model = ModelStyle()
outputs_img = model(img)
outputs_img_style = model(img_style)

gram_matrix_style = [gram_matrix(x) for x in outputs_img_style[:model.num_style_layers]]

content_weight = 1
style_weight = 1000
best_loss = -1
epochs = 100
best_img = img_create.clone()

optimizer = optim.Adam(params=[img_create], lr=0.01)

for _e in range(epochs):
    outputs_img_create = model(img_create)

    loss_content = get_content_loss(outputs_img_create[-1], outputs_img[-1])
    loss_style = get_style_loss(outputs_img_create, gram_matrix_style)
    loss = content_weight * loss_content + style_weight * loss_style

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    img_create.data.clamp(0, 1)

    if loss < best_loss or best_loss < 0:
        best_loss = loss
        best_img = img_create.clone()

x = best_img.detach().squeeze()         # Убираем batch-измерение и градиенты
low, hi = torch.min(x), torch.max(x)    # Находим min и max для нормализации
x = (x - low) / (hi - low) * 255.0      # Нормализуем в диапазон [0, 255]
x = x.permute(1, 2, 0)                  # Меняем оси: (C, H, W) -> (H, W, C)
x = x.numpy()                           # Преобразуем в numpy-массив
x = np.clip(x, 0, 255).astype('uint8')  # Отсекаем выбросы и меняем тип

image = Image.fromarray(x, 'RGB')       # Создаём объект изображения
image.save("result.jpg")                # Сохраняем в файл

plt.imshow(x)                           # Отображаем через matplotlib
plt.show()